In [4]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # expose only physical GPU 1 to this process

In [5]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import Tuple
from torch_geometric.data import Data
from torch_geometric.nn import GINConv, GINEConv
from torch_geometric.loader import NeighborLoader

In [10]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))

CUDA available: True
Visible GPU count: 1
CUDA device name: NVIDIA GeForce RTX 4090


In [11]:
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
# Hyperparameters
conv_type = "gin"
hidden_dim = 16
num_layers = 2
dropout = 0
train_eps = False

seed = 1
lr = 1e-3
weight_decay = 1e-4
epochs = 200
val_ratio = 0.1
early_stopping_patience = 30

In [13]:
train_file = "../../../data/20ng_graph/twenty_newsgroups_train_induced_graph.pt"
test_file = "../../../data/20ng_graph/twenty_newsgroups_test_induced_graph.pt"
full_file = "../../../data/20ng_graph/twenty_newsgroups_full_graph.pt"
metadata_file = "../../../data/20ng_graph/metadata.json"

In [14]:
# Cell: loaders/helpers
def safe_load_graph(path: Path) -> Data:
    try:
        data = torch.load(path, weights_only=False)  # newer PyTorch
    except TypeError:
        data = torch.load(path)  # fallback
    if not isinstance(data, Data):
        raise TypeError(f"Expected PyG Data object, got: {type(data)}")
    if not hasattr(data, "edge_weight"):
        data.edge_weight = torch.ones(data.edge_index.size(1), dtype=torch.float32)
    data.y = data.y.long()
    return data

def split_train_val(num_nodes: int, ratio: float, seed: int) -> Tuple[torch.Tensor, torch.Tensor]:
    if not (0.0 < ratio < 1.0):
        raise ValueError("val_ratio must be in (0, 1)")
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(num_nodes, generator=g)
    n_val = max(1, int(ratio * num_nodes))
    n_val = min(n_val, num_nodes - 1)
    val_idx = perm[:n_val]
    train_idx = perm[n_val:]

    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    return train_mask, val_mask

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor, mask: torch.Tensor | None = None) -> float:
    if mask is None:
        pred = logits.argmax(dim=-1)
        return (pred == y).float().mean().item()
    idx = torch.where(mask)[0]
    if idx.numel() == 0:
        return 0.0
    pred = logits[idx].argmax(dim=-1)
    return (pred == y[idx]).float().mean().item

In [15]:
class GIN(nn.Module):
    def __init__(
        self,
        num_classes,
        in_channels,
        hidden_dim=64,
        num_layers=5,      # deeper network
        dropout=0,
        train_eps=True,
        use_residual=True,
    ):
        super().__init__()
        assert num_layers >= 2

        self.dropout = dropout
        self.use_residual = use_residual

        def mlp(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Linear(out_dim, out_dim),
            )

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        # First layer
        self.convs.append(GINConv(mlp(in_channels, hidden_dim), train_eps=train_eps))
        self.norms.append(nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.convs.append(GINConv(mlp(hidden_dim, hidden_dim), train_eps=train_eps))
            self.norms.append(nn.BatchNorm1d(hidden_dim))

        # Jumping-knowledge style head: concatenate all layer outputs
        self.lin = nn.Linear(hidden_dim * num_layers, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        h_list = []

        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            h = conv(x, edge_index)
            h = norm(h)
            h = F.relu(h)

            if self.use_residual and i > 0 and h.shape == x.shape:
                h = h + x

            h = F.dropout(h, p=self.dropout, training=self.training)
            x = h
            h_list.append(h)

        x = torch.cat(h_list, dim=-1)
        x = self.lin(x)

        # NeighborLoader: first batch_size nodes are target nodes
        if hasattr(data, "batch_size"):
            x = x[:data.batch_size]
        return x

In [16]:
train_data = safe_load_graph(train_file) # no .to(device)
test_data = safe_load_graph(test_file) # no .to(device)

train_mask, val_mask = split_train_val(train_data.num_nodes, val_ratio, seed=seed)

print("Train graph:", train_data)
print("Test graph :", test_data)

Train graph: Data(x=[11314, 384], edge_index=[2, 8256453], y=[11314], edge_weight=[8256453])
Test graph : Data(x=[7532, 384], edge_index=[2, 4311614], y=[7532], edge_weight=[4311614])


In [17]:
try:
    with open(metadata_file, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    num_classes = int(metadata.get("num_classes", int(torch.cat([train_data.y]).max().item() + 1)))
except:
    num_classes = int(torch.cat([train_data.y, test_data.y]).max().item() + 1)

In [18]:
train_loader = NeighborLoader(
train_data,
input_nodes=train_mask,
num_neighbors=[15, 10, 10, 5, 5],
batch_size=1024,
shuffle=True
)

val_loader = NeighborLoader(
train_data,
input_nodes=val_mask,
num_neighbors=[15, 10, 10, 5, 5],
batch_size=2048,
shuffle=False
)

test_loader = NeighborLoader(
test_data,
input_nodes=torch.arange(test_data.num_nodes),
num_neighbors=[15, 10, 10, 5, 5],
batch_size=2048,
shuffle=False
)

model = GIN(
num_classes=num_classes,
in_channels=train_data.x.size(-1),
hidden_dim=64,
num_layers=5,
train_eps=True
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

In [19]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [20]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in loader:
        batch = batch.to(device)
        target_size = batch.batch_size
        y = batch.y[:target_size]

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(batch)[:target_size]
            loss = F.cross_entropy(logits, y)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_samples += y.size(0)

    return total_loss / total_samples, total_correct / total_samples

best_val_acc = -1.0
best_state = None

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

model.load_state_dict(best_state)
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc : {test_acc:.4f}")

Epoch 001 | train_loss=2.7321 train_acc=0.1928 | val_loss=3.1081 val_acc=0.0460
Epoch 010 | train_loss=1.5107 train_acc=0.5420 | val_loss=1.6126 val_acc=0.5128
Epoch 020 | train_loss=1.3652 train_acc=0.5825 | val_loss=1.5461 val_acc=0.5393
Epoch 030 | train_loss=1.2944 train_acc=0.5992 | val_loss=1.5247 val_acc=0.5438
Epoch 040 | train_loss=1.2182 train_acc=0.6221 | val_loss=1.4271 val_acc=0.5871
Epoch 050 | train_loss=1.1805 train_acc=0.6359 | val_loss=1.4462 val_acc=0.5738
Epoch 060 | train_loss=1.1122 train_acc=0.6513 | val_loss=1.4935 val_acc=0.5632
Epoch 070 | train_loss=1.0589 train_acc=0.6694 | val_loss=1.4197 val_acc=0.5853
Epoch 080 | train_loss=1.0204 train_acc=0.6824 | val_loss=1.4552 val_acc=0.5791
Epoch 090 | train_loss=0.9630 train_acc=0.7023 | val_loss=1.5010 val_acc=0.5809
Epoch 100 | train_loss=0.9101 train_acc=0.7127 | val_loss=1.4702 val_acc=0.5853
Epoch 110 | train_loss=0.8598 train_acc=0.7283 | val_loss=1.4340 val_acc=0.6057
Epoch 120 | train_loss=0.8271 train_acc=

In [21]:
# Evaluate best checkpoint on induced test graph
if best_state is None:
    raise RuntimeError("best_state is None. Train first so a best validation checkpoint is saved.")

# Restore best model weights
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    # test_data is on CPU in your notebook, so move it for evaluation only
    test_batch = test_data.to(device)
    test_logits = model(test_batch)

    test_loss = F.cross_entropy(test_logits, test_batch.y).item()
    test_pred = test_logits.argmax(dim=-1)
    test_acc = (test_pred == test_batch.y).float().mean().item()

print(f"Test loss: {test_loss:.4f}")
print(f"Test acc : {test_acc:.4f}")

Test loss: 181902464.0000
Test acc : 0.0523
